# NB4 — LOSO: Janela Ótima, Nível Ótimo e Comparação de Modelos

**Pipeline de Predição de Crises Epilépticas a partir de EEG**

## Fluxo

```
NB3 → data/features/{dataset}__{pat}.npz   (X_pre, X_inter, t_pre, t_inter, ctx_ids)
NB2 → data/preictal_estimate.json          (PRE_SEC individual por paciente)
                          ↓
NB4 → Etapa 1: qual janela W performa melhor?      (livre=W, fixo=nível R5/máx, modelo=RF)
       Etapa 2: qual nível de canais performa melhor?  (livre=nível, fixo=W ótima, modelo=RF)
       Etapa 3: qual modelo performa melhor?         (livre=modelo, fixo=W ótima + nível ótimo)
```

## Design experimental — LOSO por contexto de crise

Cada contexto de crise (`ctx_id`) é uma unidade de leave-one-out: o modelo treina
em todos os outros contextos do paciente (undersample 1:1 pré/inter no treino) e é
testado no contexto deixado de fora, **sem undersampling no teste**.

## Correções aplicadas nesta versão

| # | Onde | Bug | Correção |
|---|------|-----|---------|
| 1 | `select_window_nested` | `all_contexts` usava UNIÃO de cp∪ci por janela, incluindo ctx_ids que existem só em cp ou só em ci para determinadas janelas → folds com `Xi_te=[]` → `None` silencioso | Trocar por UNIÃO das **interseções** por janela |
| 2 | `_train_eval_single` | `return None` quando treino ou teste vazio apagava o fold_ctx do CSV → N_Ctx reportado < real | Retornar dict com nan + campo `erro` em vez de None |
| 3 | `select_window_nested` | Pacientes com exatamente 2 ctx eram silenciosamente excluídos (exige ≥3) → sem `PATIENT_WINDOW` → pulados nas Etapas 2 e 3 | Fallback para LOSO direto com janela default |
| 4 | `show_stage_by_dataset` | `N_Ctx_Total` calculado via `fold_ctx.nunique()` — dependia dos folds que completaram, não dos contextos reais do paciente | Carregar `pipeline_manifest.json` para N_Ctx_Total canônico |

## Melhorias aplicadas

- **Z-score por contexto** em `load_patient_features` (remove drift inter-sessão)
- **Múltiplas seeds** no undersample (`N_SEEDS = 5`, seeds 42-46)
- **Opção B — interictal completo no teste**: FP/h calculado sobre todo o interictal disponível no contexto de teste (não só W min), tornando a métrica clinicamente válida e comparável entre janelas
- **RF ajustado**: `n_estimators=200, max_depth=12, class_weight=balanced` — `min_samples_leaf` removido após teste empírico em chb06 (Config A vs B): RF mais permissivo teve AUC +0.067
- **SelectKBest desativado** (`N_FEATURES_SELECT=None`): prejudicava RF com poucos dados por fold; pode ser reativado para SVM/kNN se necessário


In [ ]:
import subprocess, sys
for pkg in ['xgboost', 'scikit-learn', 'tqdm']:
    try:
        __import__(pkg.replace('scikit-learn','sklearn'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependências OK.')

## 1. Imports e configuração

In [ ]:
import json, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from xgboost import XGBClassifier

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def tqdm(it, **kw): return it

warnings.filterwarnings('ignore')

ROOT     = Path('data')
FEAT_DIR = ROOT / 'features'
PRE_EST  = ROOT / 'preictal_estimate.json'
MANIFEST = ROOT / 'pipeline_manifest.json'
OUT_DIR  = ROOT / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FEATS_CH = 19

# ── Janelas experimentais (Etapa 1) ───────────────────────────────────────────
WINDOWS_FIXED = {
    'W1': 10 * 60,   # 10 min
    'W2': 15 * 60,   # 15 min
    'W3': 20 * 60,   # 20 min
    'W4': 25 * 60,   # 25 min
}
# W5 = PRE_SEC individual do paciente (lido do preictal_estimate.json)

# Fallback de PRE_SEC para pacientes sem estimativa válida no NB2
# Atualizar com a mediana global após novo NB2
PRESEC_FALLBACK_S = 10.2 * 60   # mediana global dos PRE_SEC válidos (NB2)

# ── Feature selection ─────────────────────────────────────────────────────────
# N_FEATURES_SELECT: número de features a manter via SelectKBest (Mutual Info)
# None = sem seleção (todas as features do nível)
# Recomendado: 60-80 para R4 (285 features → 70)
# Impacto: reduz curse of dimensionality para SVM e kNN; RF/XGB são robustos
N_FEATURES_SELECT = None  # SelectKBest desativado — prejudica RF com poucos dados

# ── Melhorias ─────────────────────────────────────────────────────────────────
ZSCORE_BY_CTX = True    # z-score por contexto antes de empilhar
N_SEEDS       = 5       # múltiplas seeds no undersample
SEEDS         = [42, 43, 44, 45, 46]

# Janela default para pacientes com apenas 2 contextos (sem seleção aninhada)
W_DEFAULT_S   = 25 * 60  # W4 = maior janela fixa (fallback para pacientes com 2 ctx)

print('Configuração carregada.')
print(f'  ZSCORE_BY_CTX = {ZSCORE_BY_CTX}')
print(f'  N_SEEDS       = {N_SEEDS}  (seeds {SEEDS})')
print(f'  W_DEFAULT_S   = {W_DEFAULT_S//60}min (fallback para pacientes com 2 ctx)')

## 2. Classificador de Distância Mínima (MDC)

In [ ]:
class MinimumDistanceClassifier:
    """Classificador de distância mínima ao centróide de classe.
    Compatível com a interface sklearn (fit/predict/predict_proba).
    """
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.centroids_ = {c: X[y == c].mean(axis=0) for c in self.classes_}
        return self

    def _dists(self, X):
        return np.column_stack([
            np.linalg.norm(X - self.centroids_[c], axis=1) for c in self.classes_
        ])

    def predict(self, X):
        d = self._dists(X)
        return self.classes_[np.argmin(d, axis=1)]

    def predict_proba(self, X):
        d = self._dists(X)
        inv = 1.0 / (d + 1e-10)
        p = inv / inv.sum(axis=1, keepdims=True)
        order = np.argsort(self.classes_)
        return p[:, order]

print('MinimumDistanceClassifier definido.')

## 3. Fábricas de modelos

In [ ]:
def make_rf():
    # n_estimators=200: bom equilíbrio velocidade/variância para este tamanho de dataset
    # max_depth=12: limita profundidade sem restringir demais (sem min_samples_leaf)
    # class_weight='balanced': compensa desbalanceamento pré/inter no teste (Opção B)
    # min_samples_leaf removido: restringia demais com folds pequenos (Config B validada)
    return RandomForestClassifier(n_estimators=200, max_depth=12,
                                  random_state=42, n_jobs=-1,
                                  class_weight='balanced')

def make_xgb():
    return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                         random_state=42, n_jobs=-1,
                         eval_metric='logloss', verbosity=0)

def make_svm():
    return SVC(kernel='rbf', C=1.0, probability=True,
               random_state=42, class_weight='balanced')

def make_lr():
    return LogisticRegression(max_iter=2000, random_state=42,
                              class_weight='balanced')

def make_nb():
    return GaussianNB()

def make_knn(k):
    return KNeighborsClassifier(n_neighbors=k)

def make_mdc():
    return MinimumDistanceClassifier()

MODELS = {
    'RF':   make_rf,
    'XGB':  make_xgb,
    'SVM':  make_svm,
    'LR':   make_lr,
    'NB':   make_nb,
    'kNN3': lambda: make_knn(3),
    'kNN5': lambda: make_knn(5),
    'kNN7': lambda: make_knn(7),
    'MDC':  make_mdc,
}

print(f'Modelos definidos: {list(MODELS.keys())}')

## 4. Carregamento dos dados por paciente

**Melhoria aplicada:** z-score por contexto antes de empilhar — remove drift
inter-sessão (impedância, ganho do amplificador) que o `StandardScaler` do fold
não conseguia corrigir sozinho.

In [ ]:
def load_patient_features(fp, zscore_by_ctx=ZSCORE_BY_CTX):
    """Carrega .npz do NB3 e retorna dict com arrays + meta.
    Se zscore_by_ctx=True, normaliza cada contexto individualmente
    antes de empilhar (remove drift inter-sessão).
    """
    raw  = np.load(fp, allow_pickle=True)
    meta = json.loads(str(raw['meta']))

    X_pre         = raw['X_pre'].copy().astype(np.float32)
    X_inter       = raw['X_inter'].copy().astype(np.float32)
    t_pre         = raw['t_pre'].copy()
    t_inter       = raw['t_inter'].copy()
    ctx_ids_pre   = raw['ctx_ids_pre'].copy()
    ctx_ids_inter = raw['ctx_ids_inter'].copy()

    if zscore_by_ctx:
        # ── Z-score por contexto: cada ctx tem sua própria média e std ──────
        # Isso remove diferenças de amplitude entre sessões de gravação
        # (impedância diferente, ganho do amplificador) sem tocar nas
        # diferenças reais entre pré-ictal e interictal DENTRO de cada ctx.
        for ctx_id in np.unique(ctx_ids_pre):
            mask = ctx_ids_pre == ctx_id
            if mask.sum() < 2: continue
            mu = X_pre[mask].mean(axis=0, keepdims=True)
            sd = X_pre[mask].std(axis=0, keepdims=True) + 1e-10
            X_pre[mask] = (X_pre[mask] - mu) / sd

        for ctx_id in np.unique(ctx_ids_inter):
            mask = ctx_ids_inter == ctx_id
            if mask.sum() < 2: continue
            mu = X_inter[mask].mean(axis=0, keepdims=True)
            sd = X_inter[mask].std(axis=0, keepdims=True) + 1e-10
            X_inter[mask] = (X_inter[mask] - mu) / sd

    return {
        'X_pre': X_pre, 'X_inter': X_inter,
        't_pre': t_pre, 't_inter': t_inter,
        'ctx_ids_pre': ctx_ids_pre, 'ctx_ids_inter': ctx_ids_inter,
        'meta': meta,
    }


# ── Carrega N_Ctx_Total canônico do manifesto (não depende de folds rodados) ──
N_CTX_CANONICAL = {}   # {(ds, pat): n_contextos_validos}
if MANIFEST.exists():
    with open(MANIFEST, encoding='utf-8') as f:
        manifest = json.load(f)
    for c in manifest.get('contexts', []):
        if c.get('npz_saved', False):
            key = (c['dataset'], c['paciente'])
            N_CTX_CANONICAL[key] = N_CTX_CANONICAL.get(key, 0) + 1
    print(f'Manifesto carregado: {len(N_CTX_CANONICAL)} pacientes, '
          f'{sum(N_CTX_CANONICAL.values())} contextos válidos')
else:
    print('AVISO: pipeline_manifest.json não encontrado — N_Ctx_Total virá dos CSVs')


# ── Carrega estimativas individuais de PRE_SEC (janela W5) ────────────────────
PRE_SEC_BY_PATIENT = {}
if PRE_EST.exists():
    with open(PRE_EST, encoding='utf-8') as f:
        pre_est_raw = json.load(f)
    for entry in pre_est_raw:
        key = (entry['dataset'], entry['paciente'])
        if entry.get('pre_sec') is not None:
            PRE_SEC_BY_PATIENT[key] = float(entry['pre_sec'])
    print(f'PRE_SEC individual carregado para {len(PRE_SEC_BY_PATIENT)} pacientes.')
else:
    print('AVISO: preictal_estimate.json não encontrado — W5 não será usada')


# ── Carrega todos os pacientes ─────────────────────────────────────────────────
PATIENTS_DATA = {}
for fp in sorted(FEAT_DIR.glob('*.npz')):
    d = load_patient_features(fp)
    ds, pat = d['meta']['dataset'], d['meta']['paciente']
    PATIENTS_DATA[(ds, pat)] = d

print(f'\n{len(PATIENTS_DATA)} pacientes carregados de {FEAT_DIR}')
for ds in sorted(set(k[0] for k in PATIENTS_DATA)):
    n = sum(1 for k in PATIENTS_DATA if k[0] == ds)
    print(f'  {ds}: {n} pacientes')

## 5. Funções centrais — filtro, LOSO, seleção de janela

### Correções nesta seção

**Correção 1** (`select_window_nested`): `all_contexts` agora usa UNIÃO das
**interseções** cp∩ci por janela. Antes usava `cp∪ci`, o que incluía ctx_ids sem
dados interictais para determinadas janelas — esses folds retornavam `None`
silenciosamente, encolhendo o N_Ctx reportado.

**Correção 2** (`_train_eval_single`): retorna `dict` com `nan` (nunca `None`)
quando treino ou teste está vazio. Assim o `fold_ctx` sempre aparece no CSV,
mesmo sem métricas válidas.

**Correção 3** (`select_window_nested` + chamada): pacientes com exatamente
2 contextos recebem fallback para `run_loso_patient` com `W_DEFAULT_S`.

**Melhoria** (`_train_eval_single`): loop de `N_SEEDS` seeds no undersample;
métricas do fold = média das seeds.

In [ ]:
def filter_window(d, W_s, level_cols=None):
    """Filtra X_pre/X_inter pelos últimos W_s segundos.
    t_pre e t_inter são negativos (referencial do fim do segmento — ver NB3).
    level_cols: None = todas as colunas; int = primeiras level_cols colunas (nível aninhado).
    """
    t_pre, t_inter = d['t_pre'], d['t_inter']
    X_pre, X_inter = d['X_pre'], d['X_inter']
    cid_pre, cid_inter = d['ctx_ids_pre'], d['ctx_ids_inter']

    mask_p = t_pre   >= -W_s
    mask_i = t_inter >= -W_s

    Xp, Xi = X_pre[mask_p], X_inter[mask_i]
    cp, ci = cid_pre[mask_p], cid_inter[mask_i]

    if level_cols is not None:
        n_cols = min(level_cols, Xp.shape[1], Xi.shape[1])
        Xp, Xi = Xp[:, :n_cols], Xi[:, :n_cols]

    return Xp, Xi, cp, ci


def get_level_cols(d, level):
    """Nº de colunas para um nível (R0..R5), respeitando disponibilidade do paciente."""
    slices = d['meta']['level_slices']
    if level in slices:
        return slices[level]
    return max(slices.values())


def _train_eval_single(Xp, Xi, cp, ci, train_contexts, test_context, model_name,
                       Xi_full=None, ci_full=None):
    """Treina em train_contexts (undersample 1:1, N_SEEDS seeds) e avalia
    em test_context (distribuição real, sem undersampling).

    Opção B: Xi_full/ci_full = interictal SEM filtro de W, usado apenas no
    teste para calcular FP/h sobre o interictal completo do contexto.
    Se None, usa Xi filtrado (comportamento original).
    """
    # ── Seleciona dados de treino ──────────────────────────────────────────────
    tr_p_mask = np.isin(cp, train_contexts)
    tr_i_mask = np.isin(ci, train_contexts)
    Xp_tr, Xi_tr = Xp[tr_p_mask], Xi[tr_i_mask]

    _base = {'fold_ctx': int(test_context), 'auc': np.nan, 'f1': np.nan,
             'sensitivity': np.nan, 'specificity': np.nan, 'fp_h': np.nan,
             'n_train': 0, 'n_test': 0}

    if len(Xp_tr) == 0 or len(Xi_tr) == 0:
        return {**_base, 'erro': 'treino_vazio'}  # CORREÇÃO 2

    # ── Seleciona dados de teste ───────────────────────────────────────────────
    te_p_mask = cp == test_context
    Xp_te = Xp[te_p_mask]

    # Opção B: interictal do teste usa o segmento COMPLETO do contexto (não filtrado por W)
    # Isso torna FP/h clinicamente válido e comparável entre janelas diferentes.
    # O treino continua usando apenas Wmin de inter (balanceado 1:1 com pré-ictal).
    if Xi_full is not None and ci_full is not None:
        te_i_mask = ci_full == test_context
        Xi_te = Xi_full[te_i_mask]
    else:
        te_i_mask = ci == test_context
        Xi_te = Xi[te_i_mask]

    if len(Xp_te) == 0 or len(Xi_te) == 0:
        return {**_base, 'n_train': len(Xp_tr)+len(Xi_tr), 'erro': 'teste_vazio'}  # CORREÇÃO 2

    X_test = np.vstack([Xp_te, Xi_te])
    y_test = np.concatenate([np.ones(len(Xp_te)), np.zeros(len(Xi_te))])

    X_test = np.nan_to_num(X_test, nan=0.0, posinf=1e6, neginf=-1e6)

    # ── Loop de seeds (MELHORIA) ──────────────────────────────────────────────
    n_min = min(len(Xp_tr), len(Xi_tr))
    seed_metrics = []

    for seed in SEEDS:
        rng   = np.random.RandomState(seed)
        idx_p = rng.choice(len(Xp_tr), n_min, replace=False)
        idx_i = rng.choice(len(Xi_tr), n_min, replace=False)

        X_train = np.vstack([Xp_tr[idx_p], Xi_tr[idx_i]])
        y_train = np.concatenate([np.ones(n_min), np.zeros(n_min)])
        X_train = np.nan_to_num(X_train, nan=0.0, posinf=1e6, neginf=-1e6)

        scaler    = StandardScaler()
        Xtr_s     = scaler.fit_transform(X_train)
        Xte_s     = scaler.transform(X_test)
        Xtr_s     = np.clip(np.nan_to_num(Xtr_s, nan=0., posinf=0., neginf=0.), -50, 50)
        Xte_s     = np.clip(np.nan_to_num(Xte_s, nan=0., posinf=0., neginf=0.), -50, 50)

        # ── Feature selection (SelectKBest + Mutual Information) ───────────
        # Remove features ruidosas/redundantes antes do fit.
        # Aplicado apenas se N_FEATURES_SELECT < n_features disponíveis.
        # Beneficia especialmente SVM e kNN (curse of dimensionality).
        # RF/XGB são robustos mas também não perdem com menos features.
        if N_FEATURES_SELECT is not None and Xtr_s.shape[1] > N_FEATURES_SELECT:
            selector = SelectKBest(mutual_info_classif, k=N_FEATURES_SELECT)
            Xtr_s    = selector.fit_transform(Xtr_s, y_train)
            Xte_s    = selector.transform(Xte_s)

        try:
            clf   = MODELS[model_name]()
            clf.fit(Xtr_s, y_train)
            y_pred = clf.predict(Xte_s)
            y_score = clf.predict_proba(Xte_s)[:, 1] if hasattr(clf,'predict_proba') else y_pred.astype(float)

            auc_ = roc_auc_score(y_test, y_score) if len(set(y_test)) > 1 else np.nan
            f1_  = f1_score(y_test, y_pred, zero_division=0)
            tn, fp_, fn, tp_ = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
            sens_ = tp_/(tp_+fn) if (tp_+fn)>0 else np.nan
            spec_ = tn/(tn+fp_) if (tn+fp_)>0 else np.nan
            dur_h = (len(Xi_te) * 15.0) / 3600.0
            fph_  = fp_/dur_h if dur_h>0 else np.nan
            seed_metrics.append({'auc':auc_,'f1':f1_,'sensitivity':sens_,
                                  'specificity':spec_,'fp_h':fph_})
        except Exception as e:
            seed_metrics.append({'auc':np.nan,'f1':np.nan,'sensitivity':np.nan,
                                  'specificity':np.nan,'fp_h':np.nan,'erro_seed':str(e)})

    # Média das seeds válidas
    def _mean(key):
        vals = [m[key] for m in seed_metrics if not np.isnan(m.get(key, np.nan))]
        return float(np.mean(vals)) if vals else np.nan

    return {
        'fold_ctx':    int(test_context),
        'auc':         _mean('auc'),
        'f1':          _mean('f1'),
        'sensitivity': _mean('sensitivity'),
        'specificity': _mean('specificity'),
        'fp_h':        _mean('fp_h'),
        'n_train':     len(X_train),
        'n_test':      len(X_test),
        'n_seeds_ok':  sum(1 for m in seed_metrics if not np.isnan(m.get('auc', np.nan))),
    }


def _run_loso_generic(Xp, Xi, cp, ci, contexts, model_name, Xi_full=None, ci_full=None):
    """LOSO restrito a um subconjunto de contextos."""
    results = []
    for held_out in contexts:
        train_contexts = [c for c in contexts if c != held_out]
        if not train_contexts:
            continue
        r = _train_eval_single(Xp, Xi, cp, ci, train_contexts, held_out, model_name,
                               Xi_full=Xi_full, ci_full=ci_full)
        if r is not None:
            results.append(r)
    return results


def run_loso_patient(d, W_s, level=None, model_name='RF'):
    """LOSO por contexto de crise para um único paciente (janela/nível fixos)."""
    level_cols = get_level_cols(d, level) if level else None
    Xp, Xi, cp, ci = filter_window(d, W_s, level_cols)
    if len(Xp) == 0 or len(Xi) == 0:
        return []
    # Apenas ctx com dados nas DUAS classes (interseção)
    contexts = sorted(set(cp) & set(ci))
    if len(contexts) < 2:
        return []
    # Opção B: passa interictal completo (sem filtro W) para o teste
    Xi_full = d['X_inter']
    ci_full = d['ctx_ids_inter']
    if level_cols is not None:
        Xi_full = Xi_full[:, :min(level_cols, Xi_full.shape[1])]
    return _run_loso_generic(Xp, Xi, cp, ci, contexts, model_name,
                             Xi_full=Xi_full, ci_full=ci_full)


def select_window_nested(d, candidate_windows, level=None, model_name='RF'):
    """Seleção de janela por paciente SEM vazamento (LOSO aninhado).

    CORREÇÃO 1: all_contexts = UNIÃO das interseções cp∩ci por janela.
    Antes usava cp∪ci, incluindo ctx sem dados inter para certas janelas.

    CORREÇÃO 3 (fallback): se all_contexts < 3, usa run_loso_patient direto
    com a maior janela disponível (pacientes com 2 ctx).

    Opção B: _Xi_full_raw/_ci_full_raw = interictal completo sem filtro W,
    usado no teste para FP/h clinicamente válido.
    """
    # Opção B: interictal completo do paciente para uso no teste
    _Xi_full_raw = d['X_inter']
    _ci_full_raw = d['ctx_ids_inter']
    level_cols = get_level_cols(d, level) if level else None
    cached = {}
    for wlabel, W_s in candidate_windows.items():
        Xp, Xi, cp, ci = filter_window(d, W_s, level_cols)
        cached[wlabel] = (Xp, Xi, cp, ci, W_s)

    # CORREÇÃO 1: UNIÃO das interseções (não UNIÃO das uniões)
    valid_per_window = [set(v[2]) & set(v[3]) for v in cached.values()]
    all_contexts = sorted(set().union(*valid_per_window))

    if len(all_contexts) < 2:
        return []

    # CORREÇÃO 3: paciente com exatamente 2 ctx → sem aninhamento, usa janela default
    if len(all_contexts) == 2:
        best_wlabel = max(candidate_windows, key=lambda k: candidate_windows[k])
        Xp, Xi, cp, ci, W_s = cached[best_wlabel]
        # Opção B: prepara inter completo com level_cols correto
        _lc = get_level_cols(d, level) if level else None
        _Xi_full = _Xi_full_raw[:, :min(_lc, _Xi_full_raw.shape[1])] if _lc else _Xi_full_raw
        results = []
        for r in _run_loso_generic(Xp, Xi, cp, ci, all_contexts, model_name,
                                   Xi_full=_Xi_full, ci_full=_ci_full_raw):
            r['chosen_window'] = best_wlabel
            r['chosen_W_s']    = W_s
            r['fallback_2ctx'] = True
            results.append(r)
        return results

    # Caso normal: >= 3 contextos → LOSO aninhado
    results = []
    for held_out in all_contexts:
        inner_contexts = [c for c in all_contexts if c != held_out]
        if len(inner_contexts) < 2:
            continue

        # Seleção interna: melhor janela usando só inner_contexts
        inner_auc = {}
        for wlabel, (Xp, Xi, cp, ci, W_s) in cached.items():
            # Filtra inner_contexts à interseção disponível para esta janela
            valid_inner = sorted(set(cp) & set(ci) & set(inner_contexts))
            if len(valid_inner) < 2:
                inner_auc[wlabel] = -np.inf
                continue
            inner_res = _run_loso_generic(Xp, Xi, cp, ci, valid_inner, model_name)
            aucs = [r['auc'] for r in inner_res if not np.isnan(r['auc'])]
            inner_auc[wlabel] = np.mean(aucs) if aucs else -np.inf

        if all(v == -np.inf for v in inner_auc.values()):
            continue
        best_wlabel = max(inner_auc, key=inner_auc.get)
        Xp, Xi, cp, ci, W_s = cached[best_wlabel]

        # Opção B: inter completo para o teste (level_cols aplicado)
        _lc = get_level_cols(d, level) if level else None
        _Xi_full = _Xi_full_raw[:, :min(_lc, _Xi_full_raw.shape[1])] if _lc else _Xi_full_raw
        r = _train_eval_single(Xp, Xi, cp, ci, inner_contexts, held_out, model_name,
                               Xi_full=_Xi_full, ci_full=_ci_full_raw)
        if r is not None:
            r['chosen_window'] = best_wlabel
            r['chosen_W_s']    = W_s
            results.append(r)
    return results


print('Funções LOSO (corridas 1-3 + melhorias) definidas.')

## 5b. Teste comparativo em chb06 — configuração atual vs alteração

Roda a **Etapa 1** isoladamente para `chb06` com duas configurações:

- **Config A (atual):** `N_FEATURES_SELECT=70`, `min_samples_leaf=5`, `n_estimators=300`
- **Config B (alteração):** `N_FEATURES_SELECT=None`, sem `min_samples_leaf`, `n_estimators=200`

Objetivo: verificar se o SelectKBest e os novos hiperparâmetros do RF prejudicam o AUC.

In [ ]:
# ── Config A: configuração ATUAL (SelectKBest k=70, min_samples_leaf=5) ──────
print("=" * 60)
print("CONFIG A — atual: N_FEATURES_SELECT=70, min_samples_leaf=5")
print("=" * 60)

# chb06 precisa estar nos dados carregados
_ds_test, _pat_test = 'CHBMIT', 'chb06'
_key_test = (_ds_test, _pat_test)

if _key_test not in PATIENTS_DATA:
    print(f"AVISO: {_pat_test} não encontrado em PATIENTS_DATA.")
    print(f"Pacientes disponíveis: {[p for _, p in PATIENTS_DATA.keys()]}")
else:
    _d_test = PATIENTS_DATA[_key_test]
    _cand   = dict(WINDOWS_FIXED)
    _cand['W5'] = PRE_SEC_BY_PATIENT.get(_key_test, PRESEC_FALLBACK_S)

    print(f"Janelas candidatas: { {k: v//60 for k,v in _cand.items()} } min")
    print(f"N_FEATURES_SELECT = {N_FEATURES_SELECT}")
    print(f"RF: n_estimators={make_rf().n_estimators}, "
          f"max_depth={make_rf().max_depth}, "
          f"min_samples_leaf={make_rf().min_samples_leaf}")
    print()

    _res_a = select_window_nested(_d_test, _cand, level=None, model_name='RF')
    _df_a  = pd.DataFrame(_res_a)

    print(f"Folds: {len(_df_a)}")
    print(_df_a[['fold_ctx','chosen_window','chosen_W_s','auc','f1',
                 'sensitivity','specificity','fp_h']].to_string(index=False))
    print()
    print(f"AUC médio  (Config A): {_df_a['auc'].mean():.4f} ± {_df_a['auc'].std():.4f}")
    print(f"F1  médio  (Config A): {_df_a['f1'].mean():.4f}")
    print(f"Sens médio (Config A): {_df_a['sensitivity'].mean():.4f}")
    print(f"Spec médio (Config A): {_df_a['specificity'].mean():.4f}")


In [ ]:
# ── Config B: alteração — sem SelectKBest, RF menos restritivo ───────────────
print("=" * 60)
print("CONFIG B — alteração: N_FEATURES_SELECT=None, sem min_samples_leaf")
print("=" * 60)

import copy as _copy
from sklearn.ensemble import RandomForestClassifier as _RFC

# Monkey-patch temporário: substitui make_rf e N_FEATURES_SELECT
_orig_make_rf          = make_rf
_orig_n_feat_select    = N_FEATURES_SELECT

def _make_rf_b():
    return _RFC(n_estimators=200, max_depth=12,
                random_state=42, n_jobs=-1,
                class_weight='balanced')

# Injeta temporariamente no escopo global
import builtins
_g = globals()
_g['make_rf']              = _make_rf_b
_g['N_FEATURES_SELECT']    = None   # desativa SelectKBest

print(f"N_FEATURES_SELECT = {N_FEATURES_SELECT}")
print(f"RF: n_estimators={_make_rf_b().n_estimators}, "
      f"max_depth={_make_rf_b().max_depth}, "
      f"min_samples_leaf={_make_rf_b().min_samples_leaf}")
print()

_cand_b = dict(WINDOWS_FIXED)
_cand_b['W5'] = PRE_SEC_BY_PATIENT.get(_key_test, PRESEC_FALLBACK_S)

_res_b = select_window_nested(PATIENTS_DATA[_key_test], _cand_b, level=None, model_name='RF')
_df_b  = pd.DataFrame(_res_b)

print(f"Folds: {len(_df_b)}")
print(_df_b[['fold_ctx','chosen_window','chosen_W_s','auc','f1',
             'sensitivity','specificity','fp_h']].to_string(index=False))
print()
print(f"AUC médio  (Config B): {_df_b['auc'].mean():.4f} ± {_df_b['auc'].std():.4f}")
print(f"F1  médio  (Config B): {_df_b['f1'].mean():.4f}")
print(f"Sens médio (Config B): {_df_b['sensitivity'].mean():.4f}")
print(f"Spec médio (Config B): {_df_b['specificity'].mean():.4f}")

# ── Comparativo lado a lado ───────────────────────────────────────────────────
print()
print("=" * 60)
print("COMPARATIVO chb06 — Config A vs Config B")
print("=" * 60)
_comp = pd.DataFrame({
    'Métrica':   ['AUC', 'F1', 'Sens', 'Spec'],
    'Config A':  [_df_a['auc'].mean(), _df_a['f1'].mean(),
                  _df_a['sensitivity'].mean(), _df_a['specificity'].mean()],
    'Config B':  [_df_b['auc'].mean(), _df_b['f1'].mean(),
                  _df_b['sensitivity'].mean(), _df_b['specificity'].mean()],
})
_comp['Δ (B-A)'] = (_comp['Config B'] - _comp['Config A']).round(4)
_comp[['Config A','Config B','Δ (B-A)']] = _comp[['Config A','Config B','Δ (B-A)']].round(4)
print(_comp.to_string(index=False))
print()
print("Δ positivo = Config B melhor | Δ negativo = Config A melhor")

# ── Restaura configuração original ────────────────────────────────────────────
_g['make_rf']           = _orig_make_rf
_g['N_FEATURES_SELECT'] = _orig_n_feat_select
print()
print("Configuração original restaurada — Etapas 1/2/3 usarão Config A.")
print("Se Config B for melhor, edite a célula 3 (N_FEATURES_SELECT=None)")
print("e a célula 7 (remover min_samples_leaf) antes de rodar as etapas.")


## 6. Etapa 1 — Qual janela W performa melhor? (LOSO aninhado, sem vazamento)

Janela escolhida **por fold** via LOSO interno, sem olhar o teste.
Fixo: nível máximo disponível por paciente, modelo=RF.

In [ ]:
rows_s1 = []

for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 1 — janela'):
    candidate_windows = dict(WINDOWS_FIXED)
    # W5: PRE_SEC individual se disponível, senão mediana global (fallback)
    w5 = PRE_SEC_BY_PATIENT.get((ds, pat), PRESEC_FALLBACK_S)
    candidate_windows['W5'] = w5

    fold_results = select_window_nested(d, candidate_windows, level=None, model_name='RF')
    for r in fold_results:
        rows_s1.append({'dataset': ds, 'paciente': pat, **r})

df_s1 = pd.DataFrame(rows_s1)
df_s1.to_csv(OUT_DIR / 'stage1_windows.csv', index=False)
print(f'Etapa 1: {len(df_s1)} folds salvos em stage1_windows.csv')

In [ ]:
pd.set_option('display.width', 120)

agg_s1 = (df_s1.groupby('chosen_window')
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'),
                    sens_mean=('sensitivity','mean'), spec_mean=('specificity','mean'),
                    fp_h_mean=('fp_h','mean'), n_folds=('fold_ctx','count'))
               .reset_index()
               .sort_values('auc_mean', ascending=False))

print('RANKING DE JANELAS (Etapa 1)')
print('='*100)
print(f'  {"Janela":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
for _, row in agg_s1.iterrows():
    print(f'  {row["chosen_window"]:<8}'
          f'{row["auc_mean"]:.3f}±{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')

print(f'\nGlobal: AUC={df_s1["auc"].mean():.3f}±{df_s1["auc"].std():.3f}  '
      f'F1={df_s1["f1"].mean():.3f}  Sens={df_s1["sensitivity"].mean():.3f}  '
      f'Espec={df_s1["specificity"].mean():.3f}  FP/h={df_s1["fp_h"].mean():.2f}')

PATIENT_WINDOW = (df_s1.groupby(['dataset','paciente'])['chosen_W_s'].median().to_dict())

print('\nJANELA CONSENSO POR PACIENTE')
print('='*70)
for (ds, pat), W_s in sorted(PATIENT_WINDOW.items()):
    n_ctx_real = N_CTX_CANONICAL.get((ds, pat), '?')
    n_ctx_fold = df_s1[(df_s1.dataset==ds)&(df_s1.paciente==pat)]['fold_ctx'].nunique()
    fallback   = df_s1[(df_s1.dataset==ds)&(df_s1.paciente==pat)].get('fallback_2ctx', pd.Series([False])).any()
    flag = ' [2ctx-fallback]' if fallback else ''
    moda = df_s1[(df_s1.dataset==ds)&(df_s1.paciente==pat)]['chosen_window'].value_counts().idxmax()
    print(f'  {ds:<10} {pat:<12} W={W_s/60:.1f}min  ({moda})  '
          f'folds={n_ctx_fold}/{n_ctx_real}{flag}')

## 7. Etapa 2 — Qual nível de canais performa melhor?

In [ ]:
LEVELS = ['R0', 'R1', 'R2', 'R3', 'R4', 'R5']
rows_s2 = []

for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 2 — níveis'):
    W_s = PATIENT_WINDOW.get((ds, pat))
    if W_s is None:
        continue

    for level in LEVELS:
        for r in run_loso_patient(d, W_s, level=level, model_name='RF'):
            rows_s2.append({'dataset': ds, 'paciente': pat, 'level': level,
                            'W_s': W_s, **r})

df_s2 = pd.DataFrame(rows_s2)
df_s2.to_csv(OUT_DIR / 'stage2_levels.csv', index=False)
print(f'Etapa 2: {len(df_s2)} folds salvos em stage2_levels.csv')

In [ ]:
agg_s2 = (df_s2.groupby('level')
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'),
                    sens_mean=('sensitivity','mean'), spec_mean=('specificity','mean'),
                    fp_h_mean=('fp_h','mean'), n_folds=('fold_ctx','count'))
               .reset_index())
agg_s2['level'] = pd.Categorical(agg_s2['level'], categories=LEVELS, ordered=True)
agg_s2 = agg_s2.sort_values('level')

print('RANKING DE NÍVEIS (Etapa 2)')
print('='*100)
print(f'  {"Nível":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
for _, row in agg_s2.iterrows():
    print(f'  {row["level"]:<8}'
          f'{row["auc_mean"]:.3f}±{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')

BEST_LEVEL = agg_s2.loc[agg_s2['auc_mean'].idxmax(), 'level']
print(f'\nMelhor nível: {BEST_LEVEL}')

## 8. Etapa 3 — Qual modelo performa melhor?

In [ ]:
rows_s3 = []

for (ds, pat), d in tqdm(sorted(PATIENTS_DATA.items()), desc='Etapa 3 — modelos'):
    W_s = PATIENT_WINDOW.get((ds, pat))
    if W_s is None:
        continue

    level = 'R0' if ds == 'SeizeIT2' else BEST_LEVEL

    for model_name in MODELS:
        for r in run_loso_patient(d, W_s, level=level, model_name=model_name):
            rows_s3.append({'dataset': ds, 'paciente': pat, 'model': model_name,
                            'W_s': W_s, **r})

df_s3 = pd.DataFrame(rows_s3)
df_s3.to_csv(OUT_DIR / 'stage3_models.csv', index=False)
print(f'Etapa 3: {len(df_s3)} folds salvos em stage3_models.csv')

In [ ]:
ALL_MODELS = ['RF', 'XGB', 'SVM', 'LR', 'NB', 'kNN3', 'kNN5', 'kNN7', 'MDC']

agg_s3 = (df_s3.groupby(['dataset','model'])
               .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                    f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                    spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                    n_folds=('fold_ctx','count'))
               .reset_index())

print('COMPARAÇÃO DE MODELOS — AUC por dataset (Etapa 3)')
print('='*112)
header = f'  {"Dataset":<12} ' + ' '.join(f'{m:<10}' for m in ALL_MODELS) + '  Melhor'
print(header)
print('  ' + '-'*(len(header)-2))
for ds in sorted(agg_s3['dataset'].unique()):
    sub  = agg_s3[agg_s3['dataset']==ds].set_index('model')
    vals = {m: sub.loc[m,'auc_mean'] if m in sub.index else float('nan') for m in ALL_MODELS}
    best = max(vals, key=lambda k: vals[k] if not np.isnan(vals[k]) else -1)
    row  = f'  {ds:<12} ' + ' '.join(
        f'{vals[m]:<10.3f}' if not np.isnan(vals[m]) else f'{"n/a":<10}' for m in ALL_MODELS
    ) + f'  {best}'
    print(row)

print()
print('MÉTRICAS COMPLETAS POR MODELO (agregado global)')
print('='*90)
print(f'  {"Modelo":<8}{"AUC":<16}{"F1":<8}{"Sens.":<8}{"Espec.":<8}{"FP/h":<8}{"n"}')
gbl_full = (df_s3.groupby('model')
                 .agg(auc_mean=('auc','mean'), auc_std=('auc','std'),
                      f1_mean=('f1','mean'), sens_mean=('sensitivity','mean'),
                      spec_mean=('specificity','mean'), fp_h_mean=('fp_h','mean'),
                      n_folds=('fold_ctx','count'))
                 .reindex(ALL_MODELS)
                 .sort_values('auc_mean', ascending=False))
for m, row in gbl_full.iterrows():
    print(f'  {m:<8}'
          f'{row["auc_mean"]:.3f}±{row["auc_std"]:.3f}   '
          f'{row["f1_mean"]:<8.3f}{row["sens_mean"]:<8.3f}'
          f'{row["spec_mean"]:<8.3f}{row["fp_h_mean"]:<8.2f}{int(row["n_folds"])}')

gbl = df_s3.groupby('model')['auc'].mean()
best_global = gbl.dropna().idxmax()
print(f'\nMelhor modelo global: {best_global}  (AUC = {gbl[best_global]:.3f})')

## 9. Resumo final e exportação

In [ ]:
summary = {
    'versao': 'NB4-v4',
    'correcoes': ['select_window_nested-uniao-intersecoes',
                  '_train_eval_single-retorna-dict-nan',
                  'select_window_nested-fallback-2ctx',
                  'N_Ctx_Total-do-manifesto'],
    'melhorias': ['zscore_por_contexto', f'undersample_{N_SEEDS}_seeds'],
    'nested_window_selection': True,
    'best_level': str(BEST_LEVEL),
    'best_model_global': str(best_global),
    'best_model_auc_global': float(gbl[best_global]),
    'patient_windows_min': {f'{k[0]}/{k[1]}': round(v/60,1)
                            for k, v in PATIENT_WINDOW.items()},
    'stage1_ranking_by_chosen_window': agg_s1.to_dict(orient='records'),
    'stage2_ranking_by_level': agg_s2.to_dict(orient='records'),
    'stage3_ranking_by_dataset_model': agg_s3.to_dict(orient='records'),
    'stage3_global_by_model': gbl_full.reset_index().to_dict(orient='records'),
}

SUMMARY_PATH = OUT_DIR / 'nb4_summary.json'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=str)

print('RESUMO FINAL')
print('='*50)
print(f'  Nível ótimo : {BEST_LEVEL}')
print(f'  Modelo ótimo: {best_global}  (AUC={gbl[best_global]:.3f})')
print(f'\nResultados em: {OUT_DIR}/')

try:
    from google.colab import files
    files.download(str(SUMMARY_PATH))
except ImportError:
    print(f'Arquivo local: {SUMMARY_PATH.resolve()}')

## 10. Tabelas por paciente

**N_Ctx_Total** agora usa o manifesto do NB1 como fonte canônica (não o `fold_ctx.nunique()`
do CSV). Isso separa claramente "quantos contextos o paciente tem" de "quantos folds
completaram" — a coluna `N_Ctx` continua mostrando os folds completados.

In [ ]:
from IPython.display import display, HTML

RES_DIR = Path('data/results')
pd.set_option('display.max_rows', None)

def show_stage_by_dataset(csv_name, group_col, title):
    fp = RES_DIR / csv_name
    if not fp.exists():
        print(f'{csv_name} não encontrado.')
        return
    df = pd.read_csv(fp)

    # CORREÇÃO 4: N_Ctx_Total do manifesto
    if N_CTX_CANONICAL:
        df['N_Ctx_Total'] = df.apply(
            lambda r: N_CTX_CANONICAL.get((r['dataset'], r['paciente']), None), axis=1)
    else:
        # Fallback: conta fold_ctx únicos por paciente (comportamento antigo)
        n_ctx_csv = df.groupby(['dataset','paciente'])['fold_ctx'].nunique().rename('N_Ctx_Total')
        df = df.merge(n_ctx_csv.reset_index(), on=['dataset','paciente'])

    agg = (df.groupby(['dataset', 'paciente', group_col])
             .agg(AUC=('auc','mean'), AUC_std=('auc','std'),
                  F1=('f1','mean'), Sens=('sensitivity','mean'),
                  Espec=('specificity','mean'), FP_h=('fp_h','mean'),
                  N_Ctx=('fold_ctx','nunique'),
                  N_Ctx_Total=('N_Ctx_Total','first'))
             .reset_index()
             .rename(columns={group_col: group_col.capitalize(), 'paciente': 'Paciente'}))

    display(HTML(f'<h3>{title}</h3>'))
    for ds in sorted(agg['dataset'].unique()):
        sub = (agg[agg['dataset']==ds]
                  .drop(columns='dataset')
                  .sort_values(['Paciente', group_col.capitalize()])
                  .reset_index(drop=True))
        cols = ['Paciente', 'N_Ctx_Total', group_col.capitalize(),
                'N_Ctx', 'AUC', 'AUC_std', 'F1', 'Sens', 'Espec', 'FP_h']
        sub = sub[[c for c in cols if c in sub.columns]]
        display(HTML(f'<h4>{ds}</h4>'))
        styled = (sub.style
                     .background_gradient(subset=['AUC'], cmap='RdYlGn', vmin=0.4, vmax=1.0)
                     .format({'AUC':'{:.3f}','AUC_std':'{:.3f}','F1':'{:.3f}',
                              'Sens':'{:.3f}','Espec':'{:.3f}','FP_h':'{:.2f}'})
                     .set_caption(ds))
        display(styled)


show_stage_by_dataset('stage1_windows.csv', 'chosen_window',
    'ETAPA 1 — Desempenho por paciente (janela escolhida via LOSO aninhado)')
show_stage_by_dataset('stage2_levels.csv', 'level',
    'ETAPA 2 — Desempenho por paciente (nível de canais)')
show_stage_by_dataset('stage3_models.csv', 'model',
    'ETAPA 3 — Desempenho por paciente (modelo)')